# E4: Data Exposure Level, plus Hybrid vs. Pure-LLM Baseline

Varies **one factor**: the personal data the phrasing LLM sees (PRD §8.1, FR-43). The reference is the **E3 winner**.

| Arm | What the LLM sees |
|---|---|
| e4-raw | user message and persona context verbatim |
| e4-pseudonymised | direct identifiers (names, client labels, IBAN, e-mail) replaced |
| e4-aggregated | personal amounts and ages as bands |
| e4-none | a depersonalised question plus Deterministic Core results only |

The Core always computes on the full data. Only what reaches the LLM changes. The filter is in `finlab/exposure.py`.

After E4, the notebook runs the **hybrid vs. pure-LLM baseline** comparison with the final Reference Configuration. In the baseline there are no Core results, no classification, no compliance prompt and no retrieval, and all checks run in Shadow Mode only. It also exports the **SM-1b manual audit sample** of at least 50 released answers.

## 1. Setup

**Runtime:** *Runtime → Change runtime type → T4 GPU*. Use the same GPU type for **all** comparative runs. A different GPU or runtime changes seeded outputs.

**Data:** upload the whole `experiments` folder of the repository to Google Drive as `MyDrive/FinanceTool/experiments`. Adjust `DRIVE_DIR` if you use a different path. Results are written to `experiments/results/` on Drive, so a disconnect loses at most one item; re-running a cell resumes where it stopped.

In [ ]:
# --- Parameters -------------------------------------------------------------------------
DRIVE_DIR = "/content/drive/MyDrive/FinanceTool/experiments"
EXPECTED_GPU = "T4"          # comparative runs must use the same GPU type (None = no check)
SMOKE_TEST = False           # True: run only a few items to test the setup
SMOKE_ITEMS = [1, 3, 31, 40, 41, 56, 65]
SCORE_QUALITY = True         # SM-9 answer-quality scoring with the scorer model (slower)
SCORER_MODEL = "qwen2.5:7b-instruct-q4_K_M"   # different model family than any model under test (OQ1)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
BASE = Path(DRIVE_DIR)
RESULTS = BASE / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BASE))
assert (BASE / "finlab").exists(), f"finlab package not found under {BASE}"

!pip -q install pypdf==6.21.1 huggingface_hub pyyaml

from finlab import colab
colab.check_gpu(EXPECTED_GPU)
colab.install_ollama()
colab.start_server()

## 2. Configuration and arms

In [ ]:
from finlab.pipeline import ExperimentConfig
from finlab import runner

EXPERIMENT = "E4"
PREVIOUS = "E3"            # its winner is the reference; an identical arm reuses its results
MODELS = {"llama3.1:8b-instruct-q4_K_M", "bge-m3", SCORER_MODEL}
NEEDS_RAG = True

REFERENCE = runner.load_reference(RESULTS, "E3", ExperimentConfig(id="reference-initial"))
MODELS = {REFERENCE.phrasing_model, "llama3.1:8b-instruct-q4_K_M", "bge-m3", SCORER_MODEL}
NEEDS_RAG = True                  # the Knowledge Base also verifies premises, so every arm loads it

def arm(id, **kw):
    d = REFERENCE.as_dict(); d.update(id=id, **kw)
    return ExperimentConfig.from_dict(d)

ARMS = {
    "e4-raw":           arm("e4-raw", data_exposure="raw"),
    "e4-pseudonymised": arm("e4-pseudonymised", data_exposure="pseudonymised"),
    "e4-aggregated":    arm("e4-aggregated", data_exposure="aggregated"),
    "e4-none":          arm("e4-none", data_exposure="none"),
}

## 3. Models, Knowledge Base, test set

In [ ]:
import datetime as dt, json
import pandas as pd
from finlab.llm import Ollama, Decoding
from finlab.pipeline import ExperimentConfig, Lab
from finlab.rag import KnowledgeBase
from finlab import runner, metrics

client = Ollama()
for m in sorted(MODELS):
    colab.pull(m)
print(colab.write_or_check_lock(RESULTS / "models_lock.json", sorted(MODELS)))

kb = None
if NEEDS_RAG:
    kb = KnowledgeBase.build(BASE / "rag_corpus", client, "bge-m3", RESULTS / "cache")
    print("KB snapshot:", kb.snapshot_id, "| passages:", len(kb.passages), "| documents:", len(kb.titles))

lab = Lab(BASE, client, nlu_model="llama3.1:8b-instruct-q4_K_M", results_dir=RESULTS, kb=kb)
items = runner.load_items(BASE, include_redteam=True, ids=SMOKE_ITEMS if SMOKE_TEST else None)
print(len(items), "items;", "rule snapshot", lab.rules.snapshot_id)

## 4. Run all arms

In [ ]:
summary = {}
for arm_id, cfg in ARMS.items():
    print(f"\n=== {arm_id} ===")
    # An arm identical to the previous winner is not run again; its results are copied.
    reused = runner.reuse_winner(lab, cfg, items, RESULTS, EXPERIMENT, PREVIOUS, SCORER_MODEL, SCORE_QUALITY)
    if reused is not None:
        summary[arm_id] = reused
        continue
    records = runner.run_arm(lab, cfg, items, RESULTS, EXPERIMENT)
    summary[arm_id] = runner.finish_arm(lab, records, RESULTS, EXPERIMENT, arm_id, SCORER_MODEL, SCORE_QUALITY)

winner, rows = metrics.select_winner(summary)
table = pd.DataFrame(rows).set_index("arm")
display(table)
print("Proposed winner:", winner)

## Decision

`select_winner` proposes a winner. Arms that exceed a counter-metric tolerance (SM-C1 ≤ 10 %, SM-C2 ≤ 15 %) are excluded. Among the rest, the order is: fewest guardrail violations (SM-3), then highest behavior accuracy (SM-4), then lowest draft mismatch rate (SM-1a), then highest answer quality (SM-9).

Check the table and the per-item view below. If you choose differently, set `CHOSEN` and give the reason in `CHOICE_NOTE`. The next experiment reads `results/<experiment>/winner.json`.

In [ ]:
CHOSEN = winner          # or e.g. "e1-llama3-rag"
CHOICE_NOTE = "automatic proposal accepted"
runner.write_winner(RESULTS, EXPERIMENT, CHOSEN, rows, ARMS)
wj = json.loads((RESULTS / EXPERIMENT / "winner.json").read_text())
wj["note"] = CHOICE_NOTE
(RESULTS / EXPERIMENT / "winner.json").write_text(json.dumps(wj, indent=2))
print("winner.json written:", CHOSEN)

## Per-item view (mismatches between outcome and expected behavior)

In [ ]:
rows_all = []
for arm_id in ARMS:
    for line in (RESULTS / EXPERIMENT / arm_id / "items_scored.jsonl").read_text().splitlines():
        r = json.loads(line)
        rows_all.append({"arm": arm_id, "item": r["item_id"], "category": r["category"], "expected": r["expected_behavior"],
                         "class": r["class"], "outcome": r["outcome"], "route": r["route"],
                         "quality": (r.get("quality") or {}).get("score")})
per_item = pd.DataFrame(rows_all)
mism = per_item[per_item["expected"] != per_item["outcome"]]
display(mism.pivot_table(index=["item", "category", "expected"], columns="arm", values="outcome", aggfunc="first"))

## 5. Hybrid vs. pure-LLM baseline (final Reference Configuration)

The final Reference Configuration is the E4 winner. The baseline uses the same phrasing model, role prompt and exposure level, with compliance `baseline` and retrieval off.

In [ ]:
FINAL = ExperimentConfig.from_dict(json.loads((RESULTS / "E4" / "winner.json").read_text())["config"])
BASELINE = ExperimentConfig.from_dict({**FINAL.as_dict(), "id": "baseline-pure-llm", "compliance": "baseline",
                                       "retrieval": False})
COMPARE = {"hybrid-final": ExperimentConfig.from_dict({**FINAL.as_dict(), "id": "hybrid-final"}),
           "baseline-pure-llm": BASELINE}
cmp_summary = {}
for arm_id, cfg in COMPARE.items():
    reused = runner.reuse_winner(lab, cfg, items, RESULTS, "HYBRID_VS_BASELINE", "E4", SCORER_MODEL, SCORE_QUALITY)
    if reused is not None:
        cmp_summary[arm_id] = reused
        continue
    recs = runner.run_arm(lab, cfg, items, RESULTS, "HYBRID_VS_BASELINE")
    cmp_summary[arm_id] = runner.finish_arm(lab, recs, RESULTS, "HYBRID_VS_BASELINE", arm_id, SCORER_MODEL, SCORE_QUALITY)
display(pd.DataFrame(cmp_summary).T)

## 6. SM-1b manual audit sample

This writes `results/HYBRID_VS_BASELINE/sm1b_audit_sample.csv` with at least 50 released answers from the final configuration. Mark every answer that contains an unsourced or wrong number. The target is 0 %.

In [ ]:
final_recs = [json.loads(l) for l in (RESULTS / "HYBRID_VS_BASELINE" / "hybrid-final" / "items_scored.jsonl").read_text().splitlines()]
runner.audit_sample(final_recs, RESULTS / "HYBRID_VS_BASELINE" / "sm1b_audit_sample.csv", n=50)
print("written; note: the Q&A set has 65 items, so the sample can contain at most the non-escalated ones")

## 7. Final configuration for the application (AD-25 handoff)

The winning configuration is written as `results/agent_config_derived.yaml`. Take it over into `config/agent/reference.yaml` of the Spring Boot app, together with its `derivedFrom` block.

In [ ]:
import yaml
derived = {
    "id": "reference",
    "llm": {"phrasing": FINAL.phrasing_model, "nlu": FINAL.nlu_model, "judge": FINAL.judge_model,
            "embedding": FINAL.embed_model},
    "retrieval": FINAL.retrieval, "rolePrompt": FINAL.role_prompt, "compliance": FINAL.compliance,
    "dataExposure": FINAL.data_exposure,
    "decoding": {"temperature": FINAL.decoding.temperature, "seed": FINAL.decoding.seed,
                 "numCtx": FINAL.decoding.num_ctx, "numPredict": FINAL.decoding.num_predict},
    "derivedFrom": {"experiments": ["E1", "E2", "E3", "E4"], "resultFiles": [f"results/{e}/winner.json" for e in ("E1","E2","E3","E4")],
                    "date": dt.date.today().isoformat()},
}
(RESULTS / "agent_config_derived.yaml").write_text(yaml.safe_dump(derived, sort_keys=False))
print(yaml.safe_dump(derived, sort_keys=False))